In [2]:
import pandas as pd

In [5]:
df = pd.read_csv("output/Kmeans/speeches_with_parlinfo_kmeans.csv")

df.head()

,basepk,speechtext,speakername,year,speechdate,speakerposition,maintopic,speechtext_oringinal,speakerparty,speechtext_word_count,...,date_of_death,is_mp_matched,party_at_date,riding_at_date,province_at_date,party_simplified,cluster,cluster_distance,second_cluster,second_cluster_distance
0,2198843,member may note receiv copi hansard deliv morn...,Marcel Joseph Aimé Lambert,1963,1963-01-22,Speaker of the House of Commons,BUSINESS OF THE HOUSE,Hon. members may have noted that they have not...,Progressive Conservative,78,...,2000-09-24,True,Progressive Conservative Party,Edmonton West,Alberta,Conservative/Progressive Conservative,3,0.987918,7,0.997085
1,2198851,wish tabl bilingu text instrument adopt sessio...,Michael Starr,1963,1963-01-22,Minister of Labour,INTERNATIONAL LABOUR CONFERENCE,I wish to table the bilingual texts of the ins...,Progressive Conservative,132,...,2000-03-16,True,Progressive Conservative Party,Ontario,Ontario,Conservative/Progressive Conservative,9,0.978530,3,0.982880
2,2199017,chairman committe whole bound admiss reject ev...,Gordon Campbell Chown,1963,1963-01-22,Deputy Speaker and Chair of Committees of the ...,MILDRED DAWSON MEAKINS,The chairman and the committee of the whole ar...,Progressive Conservative,57,...,2002-07-31,True,Progressive Conservative Party,Winnipeg South,Manitoba,Conservative/Progressive Conservative,16,0.968756,3,0.993059
3,2199034,way affect case may valid file find follow que...,Gordon Campbell Chown,1963,1963-01-22,Deputy Speaker and Chair of Committees of the ...,MILDRED DAWSON MEAKINS,This will not in any way adversely affect your...,Progressive Conservative,153,...,2002-07-31,True,Progressive Conservative Party,Winnipeg South,Manitoba,Conservative/Progressive Conservative,3,0.983459,7,0.988222
4,2199095,mr speaker wish inform hous part continu effor...,John George Diefenbaker,1963,1963-01-23,Prime Minister; President of the Privy Council,* PENSIONS,"Mr. Speaker, I wish to inform the house that a...",Progressive Conservative,129,...,1979-08-16 (Died in Office),True,Progressive Conservative Party,Prince Albert,Saskatchewan,Conservative/Progressive Conservative,10,0.983564,7,0.988985


In [8]:
df["distance_diff"] = - df["cluster_distance"] + df["second_cluster_distance"]
df["distance_diff"].describe()

KeyError: 'cluster_distance'

In [9]:
df = pd.read_csv("output/speeches_with_parlinfo.csv")
df.describe()

,basepk,year,speechtext_word_count,speech_length_words,person_id
count,1.220540e+05,122054.000000,122054.000000,122054.000000,121040.000000
mean,3.179297e+06,1978.758844,153.658954,153.658954,882.075578
std,5.085280e+05,8.730967,172.123370,172.123370,494.608770
min,2.198843e+06,1963.000000,50.000000,50.000000,1.000000
25%,2.766076e+06,1971.000000,62.000000,62.000000,440.000000
50%,3.217122e+06,1979.000000,85.000000,85.000000,920.000000
75%,3.639866e+06,1987.000000,161.000000,161.000000,1270.000000
max,3.952199e+06,1993.000000,1147.000000,1147.000000,1765.000000


In [ ]:
list = df['is_mp_matched'].tolist()
s = set(list)
print(s)

{False, True}


In [4]:
# =============================================================================
# Share of high disagreement within each category
# =============================================================================

from pathlib import Path
import csv

import pandas as pd


# =============================================================================
# Path settings
# =============================================================================

# temp.ipynb 位于 work_1953_1993 文件夹中
WORK_DIR = Path.cwd()

INPUT_FILE = WORK_DIR / "output" / "Speeches_final.csv"


# =============================================================================
# Column settings
# =============================================================================

CATEGORY_COLUMN = "category"
DISAGREEMENT_COLUMN = "disagreement"

THRESHOLD = 0.5

REQUIRED_COLUMNS = [
    CATEGORY_COLUMN,
    DISAGREEMENT_COLUMN,
]


# =============================================================================
# Load selected columns only
# =============================================================================

def load_selected_columns_streaming(input_file, usecols, encoding="utf-8-sig"):
    """
    Stream-read selected columns from a large CSV file.
    This avoids loading large text columns such as speechtext.
    """
    if not input_file.exists():
        raise FileNotFoundError(f"Input file was not found: {input_file}")

    rows = []

    with open(input_file, mode="r", encoding=encoding, newline="") as file:
        reader = csv.reader(file)

        header = next(reader)
        header = [column.strip("\ufeff") for column in header]

        missing_columns = [
            column for column in usecols
            if column not in header
        ]

        if missing_columns:
            raise ValueError(
                f"Columns were not found in the input file: {missing_columns}"
            )

        column_indices = [
            header.index(column)
            for column in usecols
        ]

        for row_number, row in enumerate(reader, start=2):
            try:
                selected_row = {
                    column: row[index]
                    for column, index in zip(usecols, column_indices)
                }
                rows.append(selected_row)

            except IndexError:
                print(f"Skipped malformed row: {row_number}")

    data = pd.DataFrame(rows)

    print(f"Loaded {len(data):,} rows and {len(data.columns)} columns.")
    return data


data = load_selected_columns_streaming(
    input_file=INPUT_FILE,
    usecols=REQUIRED_COLUMNS,
)


# =============================================================================
# Clean data
# =============================================================================

data[CATEGORY_COLUMN] = (
    data[CATEGORY_COLUMN]
    .astype(str)
    .str.strip()
)

data[DISAGREEMENT_COLUMN] = pd.to_numeric(
    data[DISAGREEMENT_COLUMN],
    errors="coerce",
)

data = data.dropna(
    subset=[
        CATEGORY_COLUMN,
        DISAGREEMENT_COLUMN,
    ]
).copy()

data = data[
    data[CATEGORY_COLUMN].ne("")
].copy()


# =============================================================================
# Calculate share of disagreement >= 0.75 within each category
# =============================================================================

data["high_disagreement"] = (
    data[DISAGREEMENT_COLUMN] >= THRESHOLD
)

category_disagreement_summary = (
    data
    .groupby(CATEGORY_COLUMN)
    .agg(
        total_cases=(DISAGREEMENT_COLUMN, "size"),
        high_disagreement_cases=("high_disagreement", "sum"),
        high_disagreement_share=("high_disagreement", "mean"),
        mean_disagreement=(DISAGREEMENT_COLUMN, "mean"),
    )
    .reset_index()
)

category_disagreement_summary["high_disagreement_percent"] = (
    category_disagreement_summary["high_disagreement_share"] * 100
)

category_disagreement_summary = (
    category_disagreement_summary
    .sort_values(
        by="high_disagreement_share",
        ascending=False,
    )
    .reset_index(drop=True)
)


# =============================================================================
# Display result
# =============================================================================

display(category_disagreement_summary)

print(f"Threshold: disagreement >= {THRESHOLD}")
print("Interpretation:")
print(
    "high_disagreement_share = "
    "number of observations with disagreement >= threshold "
    "/ total valid observations within the same category"
)

Loaded 121,022 rows and 2 columns.


,category,total_cases,high_disagreement_cases,high_disagreement_share,mean_disagreement,high_disagreement_percent
0,"Justice, Rights & Immigration",8365,2505,0.299462,0.278936,29.946204
1,Defence & National Security,3565,740,0.207574,0.196143,20.757363
2,Trade & Foreign Affairs,11088,2178,0.196429,0.213812,19.642857
3,Constitutional & Intergovernmental Relations,5134,967,0.188352,0.232723,18.835216
4,Government & Electoral Affairs,12553,2296,0.182904,0.204684,18.290448
5,Social Policy,13065,1926,0.147417,0.176181,14.741676
6,Taxation,6329,863,0.136356,0.192700,13.635645
7,Parliamentary Procedure,22653,3051,0.134684,0.203154,13.468415
8,"Sectoral Policy: Resources, Agriculture & Tran...",24149,3209,0.132883,0.170127,13.288335
9,Economy & Public Finance,14121,1511,0.107004,0.173341,10.700375


Threshold: disagreement >= 0.5
Interpretation:
high_disagreement_share = number of observations with disagreement >= threshold / total valid observations within the same category


In [1]:
from pathlib import Path
import pandas as pd

# 当前 notebook 所在目录应为 work_1953_1993
input_file = Path("output") / "Speeches_final.csv"

# 读取 header 和前 5 行
df_head = pd.read_csv(input_file, nrows=5)

# 显示列名
print("Header / Columns:")
print(df_head.columns.tolist())

# 显示前 5 行
df_head

Header / Columns:
['basepk', 'speechtext', 'speakername', 'year', 'speechdate', 'speakerposition', 'maintopic', 'speechtext_oringinal', 'speakerparty', 'speechtext_word_count', 'speech_date', 'speech_length_words', 'speaker_name_key', 'speaker_last_name_key', 'person_id', 'matched_parlinfo_name', 'mp_start_date', 'mp_end_date', 'match_method', 'gender', 'date_of_birth', 'date_of_death', 'is_mp_matched', 'party_at_date', 'riding_at_date', 'province_at_date', 'party_simplified', 'topic', 'category', 'procedural_score', 'cluster', 'cluster_distance', 'second_cluster', 'second_cluster_distance', 'cluster_score', 'disagreement']


,basepk,speechtext,speakername,year,speechdate,speakerposition,maintopic,speechtext_oringinal,speakerparty,speechtext_word_count,...,party_simplified,topic,category,procedural_score,cluster,cluster_distance,second_cluster,second_cluster_distance,cluster_score,disagreement
0,2198843,member may note receiv copi hansard deliv morn...,Marcel Joseph Aimé Lambert,1963,1963-01-22,Speaker of the House of Commons,BUSINESS OF THE HOUSE,Hon. members may have noted that they have not...,Progressive Conservative,78,...,Conservative/Progressive Conservative,Hansard length and content,Parliamentary Procedure,0.50,3,0.987918,7,0.997085,0.50,0.00
1,2198851,wish tabl bilingu text instrument adopt sessio...,Michael Starr,1963,1963-01-22,Minister of Labour,INTERNATIONAL LABOUR CONFERENCE,I wish to table the bilingual texts of the ins...,Progressive Conservative,132,...,Conservative/Progressive Conservative,international labour conference instruments,Trade & Foreign Affairs,0.25,9,0.978530,3,0.982880,0.00,0.25
2,2199017,chairman committe whole bound admiss reject ev...,Gordon Campbell Chown,1963,1963-01-22,Deputy Speaker and Chair of Committees of the ...,MILDRED DAWSON MEAKINS,The chairman and the committee of the whole ar...,Progressive Conservative,57,...,Conservative/Progressive Conservative,jurisdiction over civil contracts,"Justice, Rights & Immigration",0.50,16,0.968756,3,0.993059,0.75,0.25
3,2199034,way affect case may valid file find follow que...,Gordon Campbell Chown,1963,1963-01-22,Deputy Speaker and Chair of Committees of the ...,MILDRED DAWSON MEAKINS,This will not in any way adversely affect your...,Progressive Conservative,153,...,Conservative/Progressive Conservative,divorce proceedings and marital agreements,"Justice, Rights & Immigration",0.50,3,0.983459,7,0.988222,0.50,0.00
4,2199095,mr speaker wish inform hous part continu effor...,John George Diefenbaker,1963,1963-01-23,Prime Minister; President of the Privy Council,* PENSIONS,"Mr. Speaker, I wish to inform the house that a...",Progressive Conservative,129,...,Conservative/Progressive Conservative,commonwealth air defence mission to India,Defence & National Security,0.00,10,0.983564,7,0.988985,0.00,0.00


In [1]:
from pathlib import Path
import pandas as pd

# 文件路径：相对于 work_1953_1993\temp.ipynb
input_file = Path("output") / "Speeches_final.csv"

# 只读取需要的两列，节省内存
df = pd.read_csv(
    input_file,
    usecols=["category", "procedural_score"],
    low_memory=False
)

# 确保 procedural_score 是数值型
df["procedural_score"] = pd.to_numeric(
    df["procedural_score"],
    errors="coerce"
)

# 按 category 计算 procedural_score 平均值
category_procedural_mean = (
    df
    .dropna(subset=["category", "procedural_score"])
    .groupby("category", as_index=False)["procedural_score"]
    .mean()
    .sort_values("procedural_score", ascending=False)
)

# 打印结果
print(category_procedural_mean)

# 在 Jupyter Notebook 中以表格形式显示
display(category_procedural_mean)

                                            category  procedural_score
5                            Parliamentary Procedure          0.649223
0       Constitutional & Intergovernmental Relations          0.209096
3                     Government & Electoral Affairs          0.201677
4                      Justice, Rights & Immigration          0.169032
8                                           Taxation          0.159401
2                           Economy & Public Finance          0.152277
9                            Trade & Foreign Affairs          0.145549
1                        Defence & National Security          0.125035
6  Sectoral Policy: Resources, Agriculture & Tran...          0.123721
7                                      Social Policy          0.121542


,category,procedural_score
5,Parliamentary Procedure,0.649223
0,Constitutional & Intergovernmental Relations,0.209096
3,Government & Electoral Affairs,0.201677
4,"Justice, Rights & Immigration",0.169032
8,Taxation,0.159401
2,Economy & Public Finance,0.152277
9,Trade & Foreign Affairs,0.145549
1,Defence & National Security,0.125035
6,"Sectoral Policy: Resources, Agriculture & Tran...",0.123721
7,Social Policy,0.121542


In [3]:
from pathlib import Path
import pandas as pd

# 文件路径：相对于 work_1953_1993\temp.ipynb
input_file = Path("output") / "Speeches_final.csv"

# 读取 category 和 disagreement 两列
df = pd.read_csv(
    input_file,
    usecols=["category", "disagreement"],
    low_memory=False
)

# 确保 disagreement 是数值型
df["disagreement"] = pd.to_numeric(
    df["disagreement"],
    errors="coerce"
)

# 清理 category
df["category"] = (
    df["category"]
    .fillna("")
    .astype(str)
    .str.strip()
)

# 去掉 category 为空、disagreement 缺失的行
df_clean = df[
    (df["category"] != "") &
    (df["disagreement"].notna())
].copy()

# 计算每个 category 的 disagreement 均值
category_disagreement_mean = (
    df_clean
    .groupby("category", as_index=False)
    .agg(
        mean_disagreement=("disagreement", "mean"),
        count=("disagreement", "size")
    )
    .sort_values("mean_disagreement", ascending=False)
)

# 显示结果
display(category_disagreement_mean)

,category,mean_disagreement,count
2,Economy & Public Finance,0.007368,14121
8,Taxation,-0.001011,6329
7,Social Policy,-0.041355,13065
0,Constitutional & Intergovernmental Relations,-0.048013,5134
6,"Sectoral Policy: Resources, Agriculture & Tran...",-0.048573,24149
9,Trade & Foreign Affairs,-0.057688,11088
3,Government & Electoral Affairs,-0.089947,12553
5,Parliamentary Procedure,-0.093416,22653
1,Defence & National Security,-0.125736,3565
4,"Justice, Rights & Immigration",-0.231656,8365
